## Define libraries

In [ ]:
import sys
import os

# Get the absolute path to the parent directory
parent_dir = os.path.abspath(os.path.join(os.getcwd(), '..'))

# Add the parent directory to sys.path if it's not already there
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

In [ ]:
from datasets import load_dataset
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

from helper import RAGHelper
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from dotenv import load_dotenv
load_dotenv()
import os
from datasets import load_dataset

import sqlite3
import pandas as pd
import re
import json

import numpy as np
from sklearn.metrics import mean_squared_error
from sklearn.metrics import f1_score

## Define Variables

In [ ]:
dataset_source = 'rungalileo/ragbench'
dataset_name = 'finqa'
data_split = 'test'

chunking_size = 1024
chunking_overlap = 200
separators = ["\n\n", "\n", " ", ".", ","]
# embedding
chromadb_folder = "database"
db_name = "finance"
persist_directory = f"{chromadb_folder}/{db_name}"
embedding_model = "BAAI/bge-base-en-v1.5"

API_KEY = "GROQ_API_KEY"

search_type = "similarity"
search_kwargs = {"k":5}

eval_model_type = "groq"
eval_model = "meta-llama/llama-4-scout-17b-16e-instruct"
eval_sample = 30

sqlite_db = "./sqldb/ragproject.db"


## Data Fetching

In [ ]:

dataset = load_dataset(dataset_source, dataset_name, split=data_split)

In [ ]:
def deduplicate_data(data):
    data_dict = {}
    for d in data:
        # print(d)
        document = " ".join(d["documents"])
        if document in data_dict:
            data_dict[document]["docid"].append(d["id"])
        else:
            # print(d)
            # break
            data_dict[document] = {"docid":[d["id"]]}
    return data_dict

In [ ]:
dedup = deduplicate_data(dataset)

In [ ]:
docs = [
    Document(
        
            metadata=v, 
            page_content=k
        
        
    )
    for k,v in dedup.items()
]

## Data Ingestion

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=chunking_size, chunk_overlap=chunking_overlap, separators=separators)
docs_chunks = text_splitter.split_documents(docs)

In [ ]:
vector_db = Chroma.from_documents(documents=docs_chunks, 
                                  embedding=HuggingFaceEmbeddings(model_name=embedding_model),
                                  persist_directory=persist_directory)

## Data Retrieval & Inference

In [ ]:
api_key = os.getenv(API_KEY)
rg = RAGHelper(api_key=api_key)


In [ ]:
retriever = vector_db.as_retriever(search_type=search_type, search_kwargs=search_kwargs)

In [ ]:
eval_message = rg.eval_message

In [ ]:
import time

start = time.time()
rg.db_insert(dataset=dataset, eval_message=eval_message, retriever=retriever,
          model_type="groq", eval_model=eval_model, sample=eval_sample)

print("Time:", time.time() - start)

## Calculating Metrics

In [ ]:
conn = sqlite3.connect(sqlite_db)
cursor = conn.cursor()

In [ ]:
df = pd.read_sql_query("select * from rag_table", conn)
# Close the existing connection
if 'conn' in globals():
    conn.close()
    print("Connection closed. The database should now be unlocked.")

In [ ]:
def parse_custom_metadata(text):
    # Remove braces and newlines
    text = text.replace('{', '').replace('}', '').replace('\n', '')
    # Split by comma
    items = text.split(',')
    meta_dict = {}
    for item in items:
        if ':' in item:
            key, value = item.split(':', 1)
            meta_dict[key.strip()] = value.strip()
    return meta_dict


In [ ]:
# 1. Apply the custom parser
df['metadata_dict'] = df['metadata'].apply(parse_custom_metadata)

# 2. Normalize and concat
metadata_df = pd.json_normalize(df['metadata_dict'])
df_final = pd.concat([df.drop(columns=['metadata', 'metadata_dict']), metadata_df], axis=1)

### filter data from database

In [ ]:
df_model = df_final[df_final['eval_model']==eval_model]

In [ ]:
df_model['relevance_gain_gpt'] = df_model['relevance_score'] - df_model['gpt_model_relevance_score']
df_model['relevance_gain_claude'] = df_model['relevance_score'] - df_model['claude_model_relevance_score']

In [ ]:

def calculate_metrics(df_llama):

    # Replace 'column1' and 'column2' with your actual column names
    rmse_relevance_gpt = np.sqrt(mean_squared_error(df_llama['gpt_model_relevance_score'], df_llama['relevance_score']))
    
    print(f"RMSE_relevance_gpt: {rmse_relevance_gpt}")
    
    rmse_relevance_claude = np.sqrt(mean_squared_error(df_llama['claude_model_relevance_score'], df_llama['relevance_score']))
    
    print(f"RMSE_relevance_claude: {rmse_relevance_claude}")
    rmse_utilization_gpt = np.sqrt(mean_squared_error(df_llama['gpt_model_utilization_score'], df_llama['utilization_score']))
    
    print(f"RMSE_utilization_gpt: {rmse_utilization_gpt}")
    
    rmse_utilization_claude = np.sqrt(mean_squared_error(df_llama['claude_model_utilization_score'], df_llama['utilization_score']))
    
    print(f"RMSE_utilization_claude: {rmse_utilization_claude}")
    rmse_completeness_gpt = np.sqrt(mean_squared_error(df_llama['gpt_model_completeness_score'], df_llama['completeness_score']))
    
    print(f"RMSE_completeness_gpt: {rmse_completeness_gpt}")
    
    rmse_completeness_claude = np.sqrt(mean_squared_error(df_llama['claude_model_completeness_score'], df_llama['completeness_score']))
    
    print(f"RMSE_completeness_claude: {rmse_completeness_claude}")

    f1score_gpt = f1_score(df_llama['gpt_model_adherence'].astype(int), df_llama['adherence'].astype(int))

    print(f"F1_adherence_gpt: {f1score_gpt}")

    f1score_claude = f1_score(df_llama['claude_model_adherence'].astype(int), df_llama['adherence'].astype(int))

    print(f"F1_adherence_claude: {f1score_claude}")
    

calculate_metrics(df_model)

In [ ]:
print(f"relevance_gain_gpt: {df_model['relevance_gain_gpt'].mean()}")
print(f"relevance_gain_claude: {df_model['relevance_gain_claude'].mean()}")